# 📥 Notebook 01 — Data Ingestion
**Urban Taxi Demand Pattern Analysis**

This notebook downloads Yellow and Green taxi trip records (.parquet) from the NYC TLC CDN and saves them locally to `data/raw/`.

---

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import requests
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from time import sleep

from utils import DATA_RAW, build_tlc_url

## 1. Configuration
Choose which taxi types, year, and months to download.

In [ ]:
# ── Configure your download scope here ────────────────────────────────────
TAXI_TYPES = ['yellow', 'green']
YEAR = 2024
MONTHS = list(range(1, 7))  # Jan–Jun 2024 (adjust as needed)

print(f"Will download {len(TAXI_TYPES) * len(MONTHS)} files")
print(f"Taxi types : {TAXI_TYPES}")
print(f"Year       : {YEAR}")
print(f"Months     : {MONTHS}")

## 2. Download Functions

In [ ]:
def download_file(url: str, save_path: Path, max_retries: int = 3) -> bool:
    """Download a file with retry logic and exponential backoff."""
    if save_path.exists():
        print(f"  ✅ Already exists: {save_path.name}")
        return True

    for attempt in range(1, max_retries + 1):
        try:
            print(f"  ⬇️  Downloading (attempt {attempt}/{max_retries}): {save_path.name}")
            response = requests.get(url, stream=True, timeout=120)
            response.raise_for_status()

            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192 * 16):
                    f.write(chunk)

            size_mb = save_path.stat().st_size / (1024 * 1024)
            print(f"  ✅ Saved: {save_path.name} ({size_mb:.1f} MB)")
            return True

        except (requests.RequestException, IOError) as e:
            print(f"  ⚠️  Attempt {attempt} failed: {e}")
            if save_path.exists():
                save_path.unlink()  # Remove partial file
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"  ⏳ Retrying in {wait}s...")
                sleep(wait)

    print(f"  ❌ Failed after {max_retries} attempts: {save_path.name}")
    return False

## 3. Download All Files

In [ ]:
DATA_RAW.mkdir(parents=True, exist_ok=True)

results = []

for taxi_type in TAXI_TYPES:
    for month in MONTHS:
        url = build_tlc_url(taxi_type, YEAR, month)
        filename = f"{taxi_type}_tripdata_{YEAR}-{month:02d}.parquet"
        save_path = DATA_RAW / filename
        success = download_file(url, save_path)
        results.append({'type': taxi_type, 'month': month, 'success': success})

print("\n" + "="*60)
print(f"Downloaded {sum(r['success'] for r in results)}/{len(results)} files successfully")

## 4. Schema Validation
Verify that each downloaded file has the expected columns.

In [ ]:
EXPECTED_COLS_YELLOW = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
    'fare_amount', 'total_amount'
]

EXPECTED_COLS_GREEN = [
    'VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
    'fare_amount', 'total_amount'
]


def validate_schema(df: pd.DataFrame, taxi_type: str) -> bool:
    """Check that the expected columns are present."""
    expected = EXPECTED_COLS_YELLOW if taxi_type == 'yellow' else EXPECTED_COLS_GREEN
    missing = [c for c in expected if c not in df.columns]
    if missing:
        print(f"  ⚠️  Missing columns: {missing}")
        return False
    return True


print("Schema Validation")
print("=" * 60)

for taxi_type in TAXI_TYPES:
    for month in MONTHS:
        filename = f"{taxi_type}_tripdata_{YEAR}-{month:02d}.parquet"
        path = DATA_RAW / filename
        if not path.exists():
            print(f"  ⏭️  Skipping (not downloaded): {filename}")
            continue
        df = pd.read_parquet(path)
        ok = validate_schema(df, taxi_type)
        status = '✅' if ok else '❌'
        print(f"  {status} {filename}  —  {len(df):,} rows, {len(df.columns)} cols")

## 5. Quick Preview
Take a quick look at the first downloaded file to sanity-check.

In [ ]:
# Preview the first available file
sample_file = DATA_RAW / f"yellow_tripdata_{YEAR}-{MONTHS[0]:02d}.parquet"
if sample_file.exists():
    df_sample = pd.read_parquet(sample_file)
    print(f"Shape: {df_sample.shape}")
    print(f"\nColumn types:\n{df_sample.dtypes}")
    print(f"\nMemory usage: {df_sample.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    df_sample.head()

---
**Next →** `02_data_cleaning.ipynb`